In [1]:
# Install dependency
!pip install keras-self-attention
from huggingface_hub import hf_hub_download
import importlib.util

# Descarga el loader unificado (está en cualquier repo)
loader_path = hf_hub_download("Reynier/dga-cnn", "dga_loader.py")
spec = importlib.util.spec_from_file_location("dga_loader", loader_path)
mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

name_model="labin"
# Carga cualquier modelo
model, m = mod.load_dga_model(name_model)
results = mod.predict_domains(m, model, ["google.com", "xkr3f9mq.ru"])
print(results)

  Preparing metadata (setup.py) ... done
  Created wheel for keras-self-attention: filename=keras_self_attention-0.51.0-py3-none-any.whl size=18895 sha256=c4216454475837ff2174041c6e320e599b3b872a69b9606d871a8d9b9aca7c47
  Stored in directory: /root/.cache/pip/wheels/9a/9d/6e/09a0f61c2edeaea9f96fecdc67f31455c363bb44a4ddabe746
Successfully built keras-self-attention


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


dga_loader.py: 0.00B [00:00, ?B/s]

LABin_best_model.keras:   0%|          | 0.00/8.43M [00:00<?, ?B/s]

model.py: 0.00B [00:00, ?B/s]

  labin ready.
[{'domain': 'google.com', 'label': 'legit', 'score': 0.0027}, {'domain': 'xkr3f9mq.ru', 'label': 'dga', 'score': 0.7069}]


In [2]:
# Para un solo dominio, pasamos el dominio en una lista
results = mod.predict_domains(m, model, ["google.com"])

# Como devuelve una lista, accedemos al primer elemento [0]
prediccion = results[0]

print(f"Dominio: {prediccion['domain']}")
print(f"Etiqueta: {prediccion['label']}")
print(f"Puntaje: {prediccion['score']}")

Dominio: google.com
Etiqueta: legit
Puntaje: 0.0027


In [3]:
import pandas as pd
import time

families = [#'bazarbackdoor.gz',
            #'bazarbackdoor_v2.gz',
            #'bazarbackdoor_v3.gz',
            #'bigviktor.gz',
            #'bumblebee.gz',
            #'ccleaner.gz',
            #'dmsniff.gz',
            #'enviserv.gz',
            #'fosniw.gz',
            #'goz.gz',
            #'gozi_rfc4343.gz',
            #'infy.gz',
            #'monerodownloader.gz',
            #'newgoz.gz',
            #'ngioweb.gz',
            #'pandabanker.gz',
            #'pizd.gz',
            #'reconyc.gz',
            'sharkbot.gz',
            'szribi.gz',
            'torpig.gz',
            'tufik.gz',
            'verblecon.gz',
            'wd.gz',
            'xshellghost.gz',
           ]

runs = 30
chunk_size = 50
offset_legit = 1500

if model is None:
    print("❌ No se pudo cargar el modelo. Deteniendo evaluación.")
else:
    print("🚀 Iniciando evaluación del modelo...")

    for family in families:
        print(f"📂 Evaluando familia: {family}")

        dga = pd.read_csv(
            f'/content/drive/My Drive/New_Families/{family}',
            chunksize=chunk_size
        )

        legit = pd.read_csv(
            '/content/drive/My Drive/Familias_Test/legit.gz',
            skiprows=range(1, offset_legit + 1),
            chunksize=chunk_size
        )

        for run in range(runs):
            print(f'  🔄 {run+1:02}/{runs}', end='\r')

            try:
                dfw = pd.concat([dga.get_chunk(), legit.get_chunk()], ignore_index=True)
            except StopIteration:
                print(f"\n⚠️ No hay suficientes datos para completar {family}")
                break

            pred = []
            prob = []
            query_time = []

            for domain_to_check in dfw.domain.values:
                st = time.time()

                try:
                    results = mod.predict_domains(m, model, [domain_to_check])
                    result = results[0]

                    label_value = 1 if result['label'] == 'dga' else 0
                    probability = result['score']

                    pred.append(label_value)
                    prob.append(probability)

                except Exception:
                    pred.append(0)
                    prob.append(0.5)

                query_time.append(time.time() - st)

            dfw['pred'] = pred
            dfw['prob'] = prob
            dfw['query_time'] = query_time

            dfw.to_csv(
                f'/content/drive/My Drive/results/results_{name_model}_{family}_{run}.csv.gz',
                index=False
            )

    print("\n✅ Evaluación completada!")

🚀 Iniciando evaluación del modelo...
📂 Evaluando familia: sharkbot.gz
📂 Evaluando familia: szribi.gz
📂 Evaluando familia: torpig.gz
📂 Evaluando familia: tufik.gz
📂 Evaluando familia: verblecon.gz
📂 Evaluando familia: wd.gz
📂 Evaluando familia: xshellghost.gz

✅ Evaluación completada!


In [4]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Función para calcular FPR y TPR
def fpr_tpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn + 1e-10)
    tpr = tp / (tp + fn + 1e-10)
    return fpr, tpr

families = ['bazarbackdoor.gz', 'bazarbackdoor_v2.gz', 'bazarbackdoor_v3.gz', 'bigviktor.gz', 'bumblebee.gz', 'ccleaner.gz', 'dmsniff.gz', 'enviserv.gz', 'fosniw.gz', 'goz.gz', 'gozi_rfc4343.gz', 'infy.gz', 'monerodownloader.gz', 'newgoz.gz', 'ngioweb.gz', 'pandabanker.gz', 'pizd.gz', 'reconyc.gz', 'sharkbot.gz', 'szribi.gz', 'torpig.gz', 'tufik.gz', 'verblecon.gz', 'wd.gz', 'xshellghost.gz']
runs = 30

# Listas para métricas globales
all_acc, all_acc_std = [], []
all_pre, all_pre_std = [], []
all_rec, all_rec_std = [], []
all_f1, all_f1_std = [], []
all_fpr, all_fpr_std = [], []
all_tpr, all_tpr_std = [], []
all_qt, all_qts = [], []
total_unknowns_global = 0

for family in families:
    acc, pre, rec, f1, fpr, tpr, qt = [], [], [], [], [], [], []
    total_unknowns = 0

    for run in range(runs):
        path = f'/content/drive/My Drive/results/results_{name_model}_{family}_{run}.csv.gz'
        if not os.path.exists(path): continue

        df = pd.read_csv(path)
        y_true = (df['label'] == 'dga').astype(int)
        y_pred = df['pred']

        acc.append(accuracy_score(y_true, y_pred))
        pre.append(precision_score(y_true, y_pred, zero_division=0))
        rec.append(recall_score(y_true, y_pred, zero_division=0))
        f1.append(f1_score(y_true, y_pred, zero_division=0))

        fpr_val, tpr_val = fpr_tpr(y_true, y_pred)
        fpr.append(fpr_val)
        tpr.append(tpr_val)
        if 'query_time' in df.columns: qt.append(df['query_time'].mean())

    if acc:
        print(f'{family.split(".")[0]:15}: '
              f'acc:{np.mean(acc):.2f}±{np.std(acc):.3f} '
              f'f1:{np.mean(f1):.2f}±{np.std(f1):.3f} '
              f'pre:{np.mean(pre):.2f}±{np.std(pre):.3f} '
              f'rec:{np.mean(rec):.2f}±{np.std(rec):.3f} '
              f'FPR:{np.mean(fpr):.2f}±{np.std(fpr):.3f} '
              f'TPR:{np.mean(tpr):.2f}±{np.std(tpr):.3f} '
              f'QT:{np.mean(qt):.5f}±{np.std(qt):.5f}')

        all_acc.append(np.mean(acc))
        all_acc_std.append(np.std(acc))
        all_pre.append(np.mean(pre))
        all_pre_std.append(np.std(pre))
        all_rec.append(np.mean(rec))
        all_rec_std.append(np.std(rec))
        all_f1.append(np.mean(f1))
        all_f1_std.append(np.std(f1))
        all_fpr.append(np.mean(fpr))
        all_fpr_std.append(np.std(fpr))
        all_tpr.append(np.mean(tpr))
        all_tpr_std.append(np.std(tpr))
        all_qt.append(np.mean(qt))
        all_qts.append(np.std(qt))
        total_unknowns_global += total_unknowns

print("\n### ✅ Métricas recolectadas correctamente ###")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
bazarbackdoor  : acc:0.59±0.034 f1:0.33±0.078 pre:0.85±0.108 rec:0.21±0.058 FPR:0.04±0.030 TPR:0.21±0.058 QT:0.15290±0.01963
bazarbackdoor_v2: acc:0.49±0.017 f1:0.02±0.028 pre:0.22±0.296 rec:0.01±0.015 FPR:0.04±0.030 TPR:0.01±0.015 QT:0.15752±0.00923
bazarbackdoor_v3: acc:0.48±0.015 f1:0.00±0.000 pre:0.00±0.000 rec:0.00±0.000 FPR:0.04±0.030 TPR:0.00±0.000 QT:0.15801±0.00704
bigviktor      : acc:0.53±0.023 f1:0.17±0.071 pre:0.75±0.153 rec:0.10±0.046 FPR:0.04±0.030 TPR:0.10±0.046 QT:0.15740±0.00728
bumblebee      : acc:0.48±0.015 f1:0.00±0.000 pre:0.00±0.000 rec:0.00±0.000 FPR:0.04±0.030 TPR:0.00±0.000 QT:0.15460±0.00673
ccleaner       : acc:0.86±0.043 f1:0.84±0.056 pre:0.95±0.035 rec:0.75±0.083 FPR:0.04±0.030 TPR:0.75±0.083 QT:0.15074±0.00734
dmsniff        : acc:0.96±0.039 f1:0.96±0.042 pre:0.96±0.028 rec:0.96±0.068 FPR:0.04±0.030 TPR:0.96±0.068 QT:0.14489±0.

In [5]:
import pandas as pd
import numpy as np

# Crear el DataFrame usando todas las listas de medias y desviaciones
metrics_dict = {
    'Family': [f.split('.')[0] for f in families],
    'Accuracy': all_acc,
    'Acc_std': all_acc_std,
    'Precision': all_pre,
    'Pre_std': all_pre_std,
    'Recall': all_rec,
    'Rec_std': all_rec_std,
    'F1-Score': all_f1,
    'F1_std': all_f1_std,
    'FPR': all_fpr,
    'FPR_std': all_fpr_std,
    'TPR': all_tpr,
    'TPR_std': all_tpr_std,
    'Query_Time_Mean': all_qt,
    'Query_Time_Std': all_qts
}

df_metrics = pd.DataFrame(metrics_dict)

# Calcular fila de promedios globales
global_means = df_metrics.mean(numeric_only=True).to_dict()
global_means['Family'] = 'GLOBAL_MEAN'

df_metrics = pd.concat([df_metrics, pd.DataFrame([global_means])], ignore_index=True)

output_path = f'/content/drive/My Drive/results/metricas_globales_final_{name_model}.csv'
df_metrics.to_csv(output_path, index=False)

print(f"✅ CSV final con todas las desviaciones guardado en: {output_path}")
display(df_metrics)

✅ CSV final con todas las desviaciones guardado en: /content/drive/My Drive/results/metricas_globales_final_labin.csv


,Family,Accuracy,Acc_std,Precision,Pre_std,Recall,Rec_std,F1-Score,F1_std,FPR,FPR_std,TPR,TPR_std,Query_Time_Mean,Query_Time_Std
0,bazarbackdoor,0.585667,0.033831,0.845240,0.108363,0.210667,0.057673,0.333848,0.077770,0.039333,0.029881,0.210667,0.057673,0.152898,0.019631
1,bazarbackdoor_v2,0.486333,0.016829,0.221746,0.296232,0.012000,0.015144,0.022449,0.028072,0.039333,0.029881,0.012000,0.015144,0.157517,0.009234
2,bazarbackdoor_v3,0.480333,0.014941,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.039333,0.029881,0.000000,0.000000,0.158006,0.007039
3,bigviktor,0.530667,0.022794,0.746081,0.152529,0.100667,0.046327,0.173151,0.071245,0.039333,0.029881,0.100667,0.046327,0.157405,0.007277
4,bumblebee,0.480333,0.014941,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.039333,0.029881,0.000000,0.000000,0.154603,0.006726
5,ccleaner,0.855333,0.042562,0.951254,0.034689,0.750000,0.082906,0.835946,0.055639,0.039333,0.029881,0.750000,0.082906,0.150742,0.007344
6,dmsniff,0.961000,0.038846,0.961038,0.028282,0.961333,0.067908,0.959959,0.041807,0.039333,0.029881,0.961333,0.067908,0.144887,0.004265
7,enviserv,0.778000,0.036551,0.939492,0.043913,0.595333,0.068641,0.726372,0.054286,0.039333,0.029881,0.595333,0.068641,0.147844,0.006304
8,fosniw,0.480333,0.014941,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.039333,0.029881,0.000000,0.000000,0.147043,0.007357
9,goz,0.980333,0.014941,0.962931,0.026995,1.000000,0.000000,0.980920,0.014181,0.039333,0.029881,1.000000,0.000000,0.143185,0.003636
